In [8]:
# From Inv python till Inv-biofuels

In [20]:
import pandas as pd

excel_file_path = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/Paper 3/Inv-python.xlsx'

years = ['2025', '2030', '2035', '2040', '2045', '2050']

# The ESM values are static (same stock every year), so we correct them by removing the accumulated investments.

# cumulativecorrected investments for each technology
cumulative_corrected_investments = {}

# a dictionary to hold the corrected investment data for each year
data_per_year = {}

for year in years:
    # the current year's data, ensuring only relevant columns are read
    current_year_data = pd.read_excel(excel_file_path, sheet_name=str(year), usecols="A:B")
    # Properly adjust column names based on the actual columns in your Excel file
    current_year_data.columns = ['Technology', 'Investments']
    
    #  'Investments' column to numeric, setting errors='coerce' to turn non-numeric values into NaN
    current_year_data['Investments'] = pd.to_numeric(current_year_data['Investments'], errors='coerce')
    
    # a list to store corrected investments for the current year
    corrected_investments = []
    
    for _, row in current_year_data.iterrows():
        tech = row['Technology']
        investment = row['Investments']
        
        # Calculate the corrected investment by subtracting the sum of previous corrected investments
        previous_sum = cumulative_corrected_investments.get(tech, 0)
        corrected_investment = investment - previous_sum if pd.notnull(investment) else 0
        
        # Ensuring the corrected investment is not negative; if it is, set it to zero
        corrected_investment = max(0, corrected_investment)
        
        # Updating the list of corrected investments for the current year
        corrected_investments.append(corrected_investment)
        
        # Only update the cumulative sum if the corrected investment is positive
        if corrected_investment > 0:
            cumulative_corrected_investments[tech] = previous_sum + corrected_investment
    
    # Updating the current year's data with the corrected investments
    current_year_data['CorrectedInvestments'] = corrected_investments
    
    # Store the updated data for the current year
    data_per_year[year] = current_year_data[['Technology', 'CorrectedInvestments']]
    
print("Done!")

Done!


In [22]:
# Assuming data_per_year contains the corrected investment data as calculated previously

# Loading the "first" sheet, which should already contain the list of all technologies and sectors
all_technologies_sectors = pd.read_excel(excel_file_path, sheet_name='first')

# a DataFrame to hold the corrected investments for each technology per year
corrected_investments_df = pd.DataFrame(all_technologies_sectors['Technology'])

# Adding corrected investments to the 'corrected_investments_df' DataFrame
for year in years:
    # Create a temporary DataFrame from 'data_per_year' for the current year
    temp_df = data_per_year[year][['Technology', 'CorrectedInvestments']]
    temp_df.rename(columns={'CorrectedInvestments': str(year)}, inplace=True)
    
    # Merging with the 'corrected_investments_df' DataFrame
    corrected_investments_df = pd.merge(corrected_investments_df, temp_df, on='Technology', how='left')

# Merging corrected investments with sectors from the "first" sheet
all_technologies_sectors = pd.merge(all_technologies_sectors[['Technology', 'Sector']], corrected_investments_df, on='Technology', how='left').fillna(0)

# Writing the updated data back to the 'first' sheet
with pd.ExcelWriter(excel_file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    all_technologies_sectors.to_excel(writer, sheet_name='first', index=False)

# Aggregati corrected investments by sector
sector_investments = all_technologies_sectors.drop(columns=['Technology']).groupby('Sector').sum()

# Writing the aggregated sector investments to the "SectorInvestments" sheet
with pd.ExcelWriter(excel_file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    sector_investments.to_excel(writer, sheet_name='Sector Investments', index=True)
print("Done!")

Done!


In [24]:
import pandas as pd
from openpyxl import load_workbook

# File paths
inv_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/Paper 3/Inv-python.xlsx'
target_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx'

# Load the source data
df_source = pd.read_excel(inv_file, sheet_name='first')

# Mapping of years to column indices in source
year_column_map = {
    2025: 'C',
    2030: 'D',
    2035: 'E',
    2040: 'F',
    2045: 'G',
    2050: 'H'
}

# Load the target workbook
wb = load_workbook(target_file)

for year, col_letter in year_column_map.items():
    sheet_name = str(year)
    
    if sheet_name not in wb.sheetnames:
        print(f"Sheet '{sheet_name}' not found in target workbook. Skipping.")
        continue

    ws = wb[sheet_name]

    # Clear old data from row 2 down
    max_row = ws.max_row
    max_col = ws.max_column
    for row in ws.iter_rows(min_row=2, max_row=max_row, max_col=max_col):
        for cell in row:
            cell.value = None

    # Get the relevant columns: A and the corresponding column (e.g., C for 2025)
    col_index = ord(col_letter) - ord('A')
    data = df_source.iloc[:, [0, col_index]].copy()  # 0 is column A
    
    # Write data to sheet starting from row 2
    for i, row in data.iterrows():
        ws.cell(row=i+2, column=1, value=row.iloc[0])  # Column A
        ws.cell(row=i+2, column=2, value=row.iloc[1])  # Column B



# Save the workbook after all updates
wb.save(target_file)
print("Sheets updated successfully.")


Sheets updated successfully.


In [26]:
# Step 1: Load the data


import pandas as pd

# File paths
input_file ='/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx'
sheet_name = 'Tech, Carrier & Sector by Year'  # Sheet containing original data
output_sheet_name = 'Energy Carrier Percentages'  # New sheet name for calculated percentages

# Step 1: Load the data
df = pd.read_excel(input_file, sheet_name=sheet_name)

# Step 2: Identify the relevant year columns (2025, 2030, 2035, 2040, 2045, 2050)
year_columns = ['2025', '2030', '2035', '2040', '2045', '2050']

# Step 3: Filter out only negative values for the calculations
# Create a DataFrame that keeps only negative values (positive values become NaN for exclusion)
negative_df = df.copy()

# Use apply instead of applymap to avoid the deprecation warning
for year in year_columns:
    negative_df[year] = negative_df[year].apply(lambda x: x if x > 0 else 0)

# Step 4: Calculate total negative input per technology for each year
total_negative_inputs = negative_df.groupby('Technology')[year_columns].transform('sum')

# Step 5: Calculate the percentage of each energy carrier input within each technology
for year in year_columns:
    # Divide only negative values by the total of negatives, ensuring no division by zero
    df[f'{year}_percent'] = (negative_df[year] / total_negative_inputs[year])

# Step 6: Save to a new sheet in the same Excel file
with pd.ExcelWriter(input_file, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    # Suppress index column in the output
    df.to_excel(writer, sheet_name=output_sheet_name, index=False)

print(f"Energy carrier percentages saved to sheet '{output_sheet_name}' in '{input_file}'.")



Energy carrier percentages saved to sheet 'Energy Carrier Percentages' in '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx'.


In [28]:
# List of years we are processing
years = [2025, 2030, 2035, 2040, 2045, 2050]

input_file ='/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx'


# Read the percentage data (assuming you already have this)
df_percentages = pd.read_excel(input_file, sheet_name='Energy Carrier Percentages')

# Print column names of the percentage DataFrame for debugging
print(f"Columns in 'Energy Carrier Percentages' sheet: {df_percentages.columns}")

# Create an empty list to collect all data
combined_results = []

# Initialize an empty DataFrame to hold all combined data for each year
final_combined_df = pd.DataFrame()

# Load the investment data for each year and calculate the final investments
for year in years:
    sheet_name = str(year)  # Sheet name corresponding to the year
    df_investments = pd.read_excel(input_file, sheet_name=sheet_name)

    # Print column names of the investment DataFrame for debugging
    print(f"Columns in '{sheet_name}' sheet: {df_investments.columns}")

    # Get the percentage for this year
    percentage_column = f"{year}_percent"

    # Merge the investments data with the percentage data (assuming both have 'Technology' column)
    df_merged = pd.merge(df_investments, df_percentages[['Technology', percentage_column, 'Energy Carrier']], on='Technology')

    # Check the columns after the merge to ensure 'Energy Carrier' is present
    print(f"Columns after merge: {df_merged.columns}")

    # Calculate the multiplied investments
    df_merged[f"{year}_investment_multiplied"] = df_merged['Investments'] * df_merged[percentage_column]

    # Select only the relevant columns: 'Technology', 'Energy Carrier', and the multiplied investment
    df_filtered = df_merged[['Technology', 'Energy Carrier', f"{year}_investment_multiplied"]]

    # Merge the data into the final combined DataFrame
    if final_combined_df.empty:
        final_combined_df = df_filtered
    else:
        final_combined_df = pd.merge(final_combined_df, df_filtered, on=['Technology', 'Energy Carrier'], how='outer')

# Save the results to a new sheet in the same Excel file
with pd.ExcelWriter(input_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    final_combined_df.to_excel(writer, sheet_name='Final_Investments_Multiplied', index=False)

print("Investment multiplications for all years saved to a single sheet in the Excel file.")


Columns in 'Energy Carrier Percentages' sheet: Index(['Technology', 'Energy Carrier', 'Sector', '2025', '2030', '2035',
       '2040', '2045', '2050', '2025_percent', '2030_percent', '2035_percent',
       '2040_percent', '2045_percent', '2050_percent'],
      dtype='object')
Columns in '2025' sheet: Index(['Technology', 'Investments'], dtype='object')
Columns after merge: Index(['Technology', 'Investments', '2025_percent', 'Energy Carrier'], dtype='object')
Columns in '2030' sheet: Index(['Technology', 'Investments'], dtype='object')
Columns after merge: Index(['Technology', 'Investments', '2030_percent', 'Energy Carrier'], dtype='object')
Columns in '2035' sheet: Index(['Technology', 'Investments'], dtype='object')
Columns after merge: Index(['Technology', 'Investments', '2035_percent', 'Energy Carrier'], dtype='object')
Columns in '2040' sheet: Index(['Technology', 'Investments'], dtype='object')
Columns after merge: Index(['Technology', 'Investments', '2040_percent', 'Energy Carrie

In [29]:
import pandas as pd

# File path for your Excel file
input_file ='/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx'

# Load the relevant sheet containing Technology, Energy Carrier, Sector, and investments
df = pd.read_excel(input_file, sheet_name="Final_Investments_Multiplied")

# Clean column names to remove extra spaces
df.columns = df.columns.str.strip()

# Inspect column names to ensure they match the expected format
print("Column names:", df.columns)

# Save only the multiplied investments back to the same sheet
with pd.ExcelWriter(input_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name='Final_Investments_Multiplied', index=False)

print("Data with only multiplied investments saved to 'Final_Investments_Multiplied'.")


Column names: Index(['Technology', 'Energy Carrier', '2025_investment_multiplied',
       '2030_investment_multiplied', '2035_investment_multiplied',
       '2040_investment_multiplied', '2045_investment_multiplied',
       '2050_investment_multiplied'],
      dtype='object')
Data with only multiplied investments saved to 'Final_Investments_Multiplied'.


In [32]:
# Sheet names
multiplied_sheet = "Final_Investments_Multiplied"  # Contains the multiplied investments
mapping_sheet = "Technology_Mapping"  # Contains the mapping of Technology to Sector

# Load the data from both sheets
df_multiplied = pd.read_excel(input_file, sheet_name=multiplied_sheet)
df_mapping = pd.read_excel(input_file, sheet_name=mapping_sheet)

# Ensure column names are clean (remove leading/trailing spaces)
df_multiplied.columns = df_multiplied.columns.str.strip()
df_mapping.columns = df_mapping.columns.str.strip()

# Inspect column names to ensure correctness
print("Loaded column names in multiplied sheet:", df_multiplied.columns)
print("Loaded column names in mapping sheet:", df_mapping.columns)

# Merge the Sector column from the mapping sheet into the multiplied investments sheet
# Assuming 'Technology' is the common column for mapping
df_result = df_multiplied.merge(df_mapping[['Technology', 'Sector']], on='Technology', how='left')

# Verify the result
print("Preview of the resulting DataFrame:")
print(df_result.head())

# Save the updated data back into the same sheet
with pd.ExcelWriter(input_file, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_result.to_excel(writer, sheet_name=multiplied_sheet, index=False)

print(f"Updated data with 'Sector' column saved back to the sheet '{multiplied_sheet}' in {input_file}.")

Loaded column names in multiplied sheet: Index(['Technology', 'Energy Carrier', '2025_investment_multiplied',
       '2030_investment_multiplied', '2035_investment_multiplied',
       '2040_investment_multiplied', '2045_investment_multiplied',
       '2050_investment_multiplied'],
      dtype='object')
Loaded column names in mapping sheet: Index(['Technology', 'Sector'], dtype='object')
Preview of the resulting DataFrame:
                                          Technology Energy Carrier  \
0    1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw)  Elektriciteit   
1    1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw)         Warmte   
2    1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw)      Waterstof   
3  1010 Micro-WKK Brandstofcel 2011 (WB bestaande...  Elektriciteit   
4  1010 Micro-WKK Brandstofcel 2011 (WB bestaande...         Warmte   

   2025_investment_multiplied  2030_investment_multiplied  \
0                    0.000000                         NaN   
1                    0.00

In [34]:
import pandas as pd

input_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx'
sheet_name = 'Final_Investments_Multiplied'  # Sheet containing original data
output_sheet_name = 'Aggregated_By_Sector'  # New sheet name for aggregated data

# Step 1: Load the data from the existing sheet
df = pd.read_excel(input_file, sheet_name=sheet_name)

# Step 2: Group by 'Sector' and 'Energy Carrier' and sum the investments for each year
year_columns = ['2025_investment_multiplied', '2030_investment_multiplied', 
                '2035_investment_multiplied', '2040_investment_multiplied', 
                '2045_investment_multiplied', '2050_investment_multiplied']

# Aggregating the investments by sector and energy carrier
aggregated_df = df.groupby(['Sector', 'Energy Carrier'])[year_columns].sum().reset_index()

# Step 3: Write the aggregated data to a new sheet without using book
with pd.ExcelWriter(input_file, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    # Directly write the DataFrame to the specified sheet
    aggregated_df.to_excel(writer, sheet_name=output_sheet_name, index=False)

print(f"Aggregated investments by sector saved to sheet '{output_sheet_name}' in '{input_file}'.")


Aggregated investments by sector saved to sheet 'Aggregated_By_Sector' in '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx'.
